# 정치 테마주 복합 데이터 분석 (EDA)

**목적**: 여론조사 지지율 변동 × 주가 × 거래량의 상관관계 분석  
**데이터**: PollStock 시스템 수집 데이터 (여론조사 + pykrx 주가)  
**분석 항목**:
1. 테마주 주가/거래량 분포 분석
2. 여론조사 지지율 변동 → 주가 반응 상관분석
3. 선거 사이클별 테마주 프리미엄 패턴
4. 당선예측 확률 → 관련주 시그널 분석
5. 종목간 상관관계 매트릭스

---

In [ ]:
import sys, json, warnings
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns

matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False
warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid', palette='muted')

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'src'))

from collectors.stock_collector import StockCollector
from collectors.poll_data_collector import PollDataCollector
from analyzers.theme_mapper import ThemeMapper
from analyzers.election_predictor import ElectionPredictor
from analyzers.stock_predictor import StockPredictor

sc = StockCollector()
tm = ThemeMapper(ROOT / 'config' / 'politician_stock_map.yaml', data_dir=str(ROOT / 'data' / 'raw'))
pdc = PollDataCollector(data_dir=str(ROOT / 'data' / 'polls'))

DAYS_UNTIL = 68  # D-68 (2026.06.03 지방선거)
print(f'PollStock EDA | D-{DAYS_UNTIL} | 추적 종목: {len(tm.get_all_tickers())}개')
print(f'여론조사 DB: {len(pdc.get_all_polls())}건')

## 1. 테마주 주가/거래량 기초 통계

In [ ]:
# 전체 테마주 60일 OHLCV 수집
tickers = tm.get_all_tickers()
stock_data = {}
for t in tickers:
    df = sc.get_ohlcv(t, days=60)
    if not df.empty and len(df) >= 10:
        stock_data[t] = df
print(f'{len(stock_data)}/{len(tickers)} 종목 데이터 수집 완료')

# 최근일 기준 스냅샷
snapshot = []
for t, df in stock_data.items():
    closes = df['종가'].values
    vols = df['거래량'].values
    returns = np.diff(closes) / closes[:-1]
    try:
        from pykrx import stock as krx_stock
        name = krx_stock.get_market_ticker_name(t)
    except:
        name = t
    snapshot.append({
        'ticker': t, 'name': name,
        'close': int(closes[-1]),
        'change_1d': round((closes[-1]/closes[-2]-1)*100, 2) if len(closes)>=2 else 0,
        'change_5d': round((closes[-1]/closes[-6]-1)*100, 2) if len(closes)>=6 else 0,
        'change_20d': round((closes[-1]/closes[-21]-1)*100, 2) if len(closes)>=21 else 0,
        'volatility': round(np.std(returns)*100, 2),
        'avg_volume_20d': int(np.mean(vols[-20:])) if len(vols)>=20 else int(np.mean(vols)),
        'volume_ratio': round(vols[-1]/np.mean(vols[-20:]), 2) if len(vols)>=20 and np.mean(vols[-20:])>0 else 0,
    })

df_snap = pd.DataFrame(snapshot).sort_values('change_1d', ascending=False)
print(f'\n=== 테마주 기초 통계 (N={len(df_snap)}) ===')
df_snap[['name','ticker','close','change_1d','change_5d','change_20d','volatility','volume_ratio']].head(15)

In [ ]:
# 등락률 분포 + 변동성 히스토그램
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1일 등락률 분포
axes[0].hist(df_snap['change_1d'], bins=20, color='#58a6ff', edgecolor='white', alpha=0.8)
axes[0].axvline(0, color='red', linestyle='--', alpha=0.5)
axes[0].set_title('1일 등락률 분포')
axes[0].set_xlabel('등락률 (%)')

# 20일 변동성 분포
axes[1].hist(df_snap['volatility'], bins=20, color='#d29922', edgecolor='white', alpha=0.8)
axes[1].set_title('변동성 (일간 수익률 표준편차) 분포')
axes[1].set_xlabel('변동성 (%)')

# 거래량 비율 분포
axes[2].hist(df_snap['volume_ratio'].clip(0, 5), bins=20, color='#3fb950', edgecolor='white', alpha=0.8)
axes[2].axvline(1, color='red', linestyle='--', alpha=0.5)
axes[2].set_title('거래량 비율 (당일/20일평균) 분포')
axes[2].set_xlabel('거래량 비율')

plt.suptitle(f'정치 테마주 기초 통계 분포 (D-{DAYS_UNTIL})', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(str(ROOT / 'data' / 'processed' / 'eda_distributions.png'), dpi=150, bbox_inches='tight')
plt.show()

## 2. 여론조사 지지율 → 주가 반응 상관분석

여론조사 DB에서 후보별 지지율 변동을 추출하고, 관련 테마주의 주가 변동과의 상관관계를 분석합니다.

In [ ]:
# 여론조사 DB 로드 + 후보별 모멘텀 계산
polls = pdc.get_all_polls()
print(f'여론조사 총 {len(polls)}건')

# 후보별 관련 종목 매핑
candidate_tickers = {}
for pol in tm.data.get('politicians', []):
    for s in pol.get('related_stocks', []):
        candidate_tickers.setdefault(pol['name'], []).append(s['ticker'])
for cand in tm.data.get('local_candidates_2026', []):
    for s in cand.get('related_stocks', []):
        candidate_tickers.setdefault(cand['name'], []).append(s['ticker'])

print(f'관련주 매핑된 후보: {len(candidate_tickers)}명')

# 모멘텀 계산
momentum_data = []
for name, tickers_list in candidate_tickers.items():
    mom = pdc.calculate_momentum(name)
    if mom.get('change') is not None:
        # 관련주 평균 등락률
        avg_change = 0
        cnt = 0
        for t in tickers_list:
            snap_row = df_snap[df_snap['ticker'] == t]
            if not snap_row.empty:
                avg_change += snap_row.iloc[0]['change_5d']
                cnt += 1
        avg_change = avg_change / cnt if cnt > 0 else 0
        
        momentum_data.append({
            'candidate': name,
            'poll_change': mom['change'],
            'poll_trend': mom['trend'],
            'current_rate': mom.get('current', 0),
            'stock_change_5d': round(avg_change, 2),
            'num_stocks': len(tickers_list),
        })

df_mom = pd.DataFrame(momentum_data)
if not df_mom.empty:
    print(f'\n=== 지지율 변동 vs 관련주 등락 (N={len(df_mom)}) ===')
    display(df_mom.sort_values('poll_change', ascending=False).head(15))
else:
    print('모멘텀 데이터 부족 (여론조사 2건 이상 필요)')

In [ ]:
# 지지율 변동 vs 주가 변동 산점도
if not df_mom.empty and len(df_mom) >= 3:
    fig, ax = plt.subplots(figsize=(10, 7))
    
    colors = df_mom['poll_change'].apply(lambda x: '#3fb950' if x > 0 else '#f85149' if x < 0 else '#8b949e')
    sizes = df_mom['num_stocks'] * 30 + 50
    
    scatter = ax.scatter(df_mom['poll_change'], df_mom['stock_change_5d'], 
                         c=colors, s=sizes, alpha=0.7, edgecolors='white', linewidth=0.5)
    
    # 후보명 레이블
    for _, row in df_mom.iterrows():
        ax.annotate(row['candidate'], (row['poll_change'], row['stock_change_5d']),
                    fontsize=8, ha='center', va='bottom', alpha=0.8)
    
    # 추세선
    if len(df_mom) >= 5:
        z = np.polyfit(df_mom['poll_change'], df_mom['stock_change_5d'], 1)
        p = np.poly1d(z)
        x_range = np.linspace(df_mom['poll_change'].min()-1, df_mom['poll_change'].max()+1, 50)
        ax.plot(x_range, p(x_range), '--', color='#58a6ff', alpha=0.6, label=f'추세선 (기울기={z[0]:.2f})')
        
        # 상관계수
        corr = df_mom['poll_change'].corr(df_mom['stock_change_5d'])
        ax.text(0.05, 0.95, f'상관계수 r = {corr:.3f}', transform=ax.transAxes,
                fontsize=11, fontweight='bold', va='top',
                bbox=dict(boxstyle='round', facecolor='#161b22', alpha=0.8, edgecolor='#30363d'))
        ax.legend()
    
    ax.axhline(0, color='gray', linestyle=':', alpha=0.4)
    ax.axvline(0, color='gray', linestyle=':', alpha=0.4)
    ax.set_xlabel('여론조사 지지율 변동 (%p)', fontsize=12)
    ax.set_ylabel('관련주 5일 등락률 (%)', fontsize=12)
    ax.set_title('여론조사 지지율 변동 → 테마주 주가 반응', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(str(ROOT / 'data' / 'processed' / 'eda_poll_stock_scatter.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('산점도 생성 불가 — 여론조사 모멘텀 데이터 부족')

## 3. 종목간 상관관계 히트맵

같은 정치인/정당 관련주끼리 동조화(co-movement)되는지 분석합니다.

In [ ]:
# 수익률 시계열 구성 (30일)
returns_dict = {}
ticker_names = {}
for t, df in stock_data.items():
    closes = df['종가'].values.astype(float)
    if len(closes) >= 20:
        rets = np.diff(closes[-31:]) / closes[-31:-1]
        returns_dict[t] = rets
        try:
            ticker_names[t] = krx_stock.get_market_ticker_name(t)
        except:
            ticker_names[t] = t

# 길이 맞추기
if len(returns_dict) >= 5:
    min_len = min(len(v) for v in returns_dict.values())
    aligned = {k: v[-min_len:] for k, v in returns_dict.items()}
    
    df_returns = pd.DataFrame(aligned)
    df_returns.columns = [ticker_names.get(c, c) for c in df_returns.columns]
    
    corr_matrix = df_returns.corr()
    
    fig, ax = plt.subplots(figsize=(14, 12))
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
    cmap = sns.diverging_palette(240, 10, as_cmap=True)
    sns.heatmap(corr_matrix, mask=mask, cmap=cmap, center=0, vmin=-1, vmax=1,
                annot=True, fmt='.2f', square=True, linewidths=0.5,
                cbar_kws={'shrink': 0.8, 'label': '상관계수'},
                annot_kws={'size': 7}, ax=ax)
    ax.set_title(f'정치 테마주 수익률 상관관계 (30일 기준, D-{DAYS_UNTIL})', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(str(ROOT / 'data' / 'processed' / 'eda_correlation_heatmap.png'), dpi=150, bbox_inches='tight')
    plt.show()
    
    # 높은 상관관계 쌍
    pairs = []
    cols = corr_matrix.columns
    for i in range(len(cols)):
        for j in range(i+1, len(cols)):
            pairs.append({'종목A': cols[i], '종목B': cols[j], '상관계수': round(corr_matrix.iloc[i,j], 3)})
    df_pairs = pd.DataFrame(pairs).sort_values('상관계수', ascending=False)
    print('=== 높은 양의 상관관계 (동조화) ===')
    display(df_pairs.head(10))
    print('\n=== 높은 음의 상관관계 (역행) ===')
    display(df_pairs.tail(5))
else:
    print('상관분석 불가 — 충분한 종목 데이터 없음')

## 4. 당선예측 모델 결과 시각화

ElectionPredictor의 지역별 당선확률을 시각화하고, 관련 테마주 영향을 분석합니다.

In [ ]:
# 당선예측 실행
ep = ElectionPredictor(pdc, tm, days_until_election=DAYS_UNTIL)
predictions = ep.predict_all_regions()
regions = predictions.get('regions', {})
print(f'당선예측: {len(regions)}개 지역 분석 완료')

# 지역별 1위 후보 당선확률 바 차트
region_data = []
for region, pred in sorted(regions.items()):
    preds = pred.get('predictions', [])
    if preds:
        leader = preds[0]
        runner = preds[1] if len(preds) > 1 else {}
        region_data.append({
            'region': region,
            'leader': leader['name'],
            'leader_party': leader.get('party', ''),
            'leader_prob': leader['win_probability'],
            'runner': runner.get('name', '-'),
            'runner_prob': runner.get('win_probability', 0),
            'gap': pred.get('gap', 0),
            'competitiveness': pred.get('competitiveness', ''),
        })

df_pred = pd.DataFrame(region_data)

fig, ax = plt.subplots(figsize=(14, 8))
party_colors = {'더불어민주당': '#1c6dd0', '국민의힘': '#e6394a'}
colors = [party_colors.get(r['leader_party'], '#888') for r in region_data]

bars = ax.barh(df_pred['region'], df_pred['leader_prob'], color=colors, alpha=0.85, edgecolor='white')
ax.barh(df_pred['region'], df_pred['runner_prob'], left=df_pred['leader_prob'],
        color=[party_colors.get(r.get('runner_party', ''), '#ccc') for r in region_data], alpha=0.3)

for i, (bar, row) in enumerate(zip(bars, region_data)):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f'{row["leader"]} {row["leader_prob"]}% ({row["competitiveness"]})',
            va='center', fontsize=8)

ax.axvline(50, color='gray', linestyle='--', alpha=0.4)
ax.set_xlabel('당선확률 (%)', fontsize=12)
ax.set_title(f'지역별 당선예측 — 1위 후보 당선확률 (D-{DAYS_UNTIL})', fontsize=14, fontweight='bold')

# 범례
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#1c6dd0', label='더불어민주당'),
                   Patch(facecolor='#e6394a', label='국민의힘')]
ax.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.savefig(str(ROOT / 'data' / 'processed' / 'eda_election_prediction.png'), dpi=150, bbox_inches='tight')
plt.show()

df_pred[['region','leader','leader_party','leader_prob','runner','runner_prob','gap','competitiveness']]

## 5. 복합 스코어 모델 (StockPredictor) 결과

In [ ]:
# 주가 예측 복합 스코어
sp = StockPredictor(sc, pdc, tm, days_until_election=DAYS_UNTIL)
sp_result = sp.analyze_all_theme_stocks(max_tickers=30)

analyses = sp_result.get('analyses', [])
print(f'분석 완료: {len(analyses)}개 종목')
print(f'사이클: {sp_result.get("cycle_phase", "")}')

# 스코어 테이블
score_data = []
for a in analyses:
    cs = a.get('composite_score', {})
    comp = cs.get('components', {})
    score_data.append({
        'name': a.get('name', a['ticker']),
        'ticker': a['ticker'],
        'total_score': cs.get('total', 0),
        'signal': cs.get('signal_kr', ''),
        'price_momentum': comp.get('price_momentum', 50),
        'volume_signal': comp.get('volume_signal', 50),
        'poll_impact': comp.get('poll_impact', 50),
        'cycle_premium': comp.get('cycle_premium', 50),
    })

df_score = pd.DataFrame(score_data).sort_values('total_score', ascending=False)

# 스코어 시각화 (레이더 차트 대안: 4요소 스택)
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# 좌: 종합 스코어 바 차트
colors_score = df_score['total_score'].apply(
    lambda x: '#3fb950' if x >= 60 else '#d29922' if x >= 40 else '#f85149')
axes[0].barh(df_score['name'], df_score['total_score'], color=colors_score, edgecolor='white', alpha=0.85)
axes[0].axvline(50, color='gray', linestyle='--', alpha=0.4)
axes[0].axvline(60, color='green', linestyle=':', alpha=0.3)
axes[0].axvline(40, color='red', linestyle=':', alpha=0.3)
axes[0].set_xlabel('복합 스코어 (0~100)')
axes[0].set_title('테마주 복합 스코어 순위')

# 우: 4요소 분해 (스택 바)
bottom = np.zeros(len(df_score))
components = [
    ('price_momentum', '주가모멘텀', '#58a6ff'),
    ('volume_signal', '거래량', '#3fb950'),
    ('poll_impact', '여론조사', '#d29922'),
    ('cycle_premium', '선거사이클', '#f85149'),
]
for col, label, color in components:
    vals = df_score[col].values * 0.25  # 각 25% 비중
    axes[1].barh(df_score['name'], vals, left=bottom, color=color, label=label, alpha=0.8)
    bottom += vals
axes[1].legend(loc='lower right', fontsize=9)
axes[1].set_xlabel('스코어 기여도')
axes[1].set_title('4요소 분해 분석')

plt.suptitle(f'정치 테마주 복합 스코어 (D-{DAYS_UNTIL} | {sp_result.get("cycle_phase", "")})',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(str(ROOT / 'data' / 'processed' / 'eda_composite_score.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f'\n=== TOP 10 Picks ===')
for p in sp_result.get('top_picks', [])[:10]:
    print(f'  {p["name"]} ({p["ticker"]}) → 스코어 {p["score"]} [{p["signal"]}] : {p["reason"]}')

## 6. 선거 사이클별 테마주 프리미엄 패턴

과거 선거 경험치 기반, D-day까지의 기대 프리미엄 곡선과 현재 위치를 시각화합니다.

In [ ]:
# 선거 사이클 프리미엄 곡선
from analyzers.stock_predictor import ELECTION_CYCLE_PATTERN

cycle_data = [
    {'phase': '초기관심\n(D-180+)', 'd_day': 180, 'premium': 0, 'volatility': 'low'},
    {'phase': '인물부상\n(D-180~120)', 'd_day': 150, 'premium': 3, 'volatility': 'medium'},
    {'phase': '경선시즌\n(D-120~60)', 'd_day': 90, 'premium': 8, 'volatility': 'high'},
    {'phase': '본선돌입\n(D-60~30)', 'd_day': 45, 'premium': 15, 'volatility': 'very_high'},
    {'phase': '막판스퍼트\n(D-30~14)', 'd_day': 22, 'premium': 20, 'volatility': 'extreme'},
    {'phase': '선거직전\n(D-14~7)', 'd_day': 10, 'premium': 12, 'volatility': 'high'},
    {'phase': 'D-day임박\n(D-7~0)', 'd_day': 3, 'premium': 5, 'volatility': 'medium'},
    {'phase': '선거후', 'd_day': -7, 'premium': -10, 'volatility': 'high'},
]

fig, ax = plt.subplots(figsize=(14, 6))

d_days = [c['d_day'] for c in cycle_data]
premiums = [c['premium'] for c in cycle_data]
phases = [c['phase'] for c in cycle_data]

# 프리미엄 곡선
ax.fill_between(d_days, premiums, alpha=0.2, color='#58a6ff')
ax.plot(d_days, premiums, 'o-', color='#58a6ff', linewidth=2.5, markersize=8, label='기대 프리미엄')

# 현재 위치 표시
current_phase = sp._get_cycle_phase()
ax.axvline(DAYS_UNTIL, color='#d29922', linewidth=2, linestyle='--', label=f'현재 D-{DAYS_UNTIL}')
ax.annotate(f'현재\nD-{DAYS_UNTIL}\n({current_phase["label"]})',
            xy=(DAYS_UNTIL, current_phase['avg_premium']),
            xytext=(DAYS_UNTIL+15, current_phase['avg_premium']+5),
            fontsize=10, fontweight='bold', color='#d29922',
            arrowprops=dict(arrowstyle='->', color='#d29922'))

# 위험 구간 강조
ax.axhspan(15, 25, alpha=0.1, color='red', label='과열 구간')
ax.axhspan(-15, -5, alpha=0.1, color='blue', label='선거 후 조정')

for i, (d, p, phase) in enumerate(zip(d_days, premiums, phases)):
    ax.annotate(phase, (d, p), textcoords='offset points', xytext=(0, 12),
                fontsize=7, ha='center', color='#8b949e')

ax.set_xlabel('D-day (선거까지 남은 일수)', fontsize=12)
ax.set_ylabel('기대 프리미엄 (%)', fontsize=12)
ax.set_title('선거 사이클별 정치 테마주 기대 프리미엄 곡선', fontsize=14, fontweight='bold')
ax.invert_xaxis()
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(ROOT / 'data' / 'processed' / 'eda_election_cycle.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f'현재 위치: D-{DAYS_UNTIL} ({current_phase["label"]})')
print(f'기대 프리미엄: {current_phase["avg_premium"]}% | 변동성: {current_phase["volatility"]}')

## 7. 종합 요약

이 노트북에서 분석한 내용:
1. **테마주 분포**: 등락률, 변동성, 거래량 비율의 기초 통계
2. **지지율-주가 상관**: 여론조사 변동과 관련주 등락의 상관계수
3. **종목간 상관관계**: 테마주끼리의 동조화/역행 패턴
4. **당선예측**: 지역별 1위 후보 당선확률과 경합도
5. **복합 스코어**: 주가모멘텀(30%) + 거래량(20%) + 여론조사(30%) + 선거사이클(20%)
6. **선거 사이클**: 현재 위치 기반 기대 프리미엄 곡선

> 데이터가 축적될수록 모델 정확도가 높아집니다. 매일 스크리닝 실행으로 시계열 데이터를 쌓아가세요.